In [6]:
# RAG(Retriever Augmentation Generation) 시스템 및 모델 파이프라인 구축(데이터 수집기 + Qdrant 의미 기반 검색엔진 적용)
# - Retrieval 단계: Qdrant 의미 기반 검색엔진을 통해 관련 문서를 빠르게 찾아냄
# - Augmentation 단계: 검색된 문서를 기반으로 LLM 입력 프롬프트를 강화 → 맥락 있는 답변 생성
# - Generation 단계: LLM이 최종 답변을 생성 → 단순 생성보다 정확성과 신뢰성이 높아짐
# - 예시) 구글, 네이버에서 사용되는 RAG 시스템 구현

# 학습 목표 - 실무에서 사용되는 파이프라인 이해 및 적용
# 1. 데이터 수집기
# - 수집 대상 도메인: Google News RSS
# - RSS 피드 파싱: feedparser 라이브러리로 RSS XML을 읽고 기사 항목 추출
# - 데이터 정제: title,link,description,pubDate,soure 필드 추출, pubDate -> Python datetime 변화
# - 수집 데이터 -> PostgreSQL 테이블 news_articles 에 Insert
# - DB 조회 + 로깅 설정으로 데이터 관리

# 2. Qdrant 의미 기반 검색 구축
# - Qdrant 구축 + 뉴스 컬렉션 생성
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

# 3. 임베딩 생성 후 Qdrant Collection에 Insert/Update
# - 임베딩 모델 로드: 온라인, 오프라인 사용
# - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# 4. Qdrant 의미 기반 검색
# - 의미 기반 검색으로 관련 문서 조회

# 5. QA 처리
# - Qdrant 검색 결과 -> QA 토크나이저 + QA 모델
# - 형태소 분석으로 한국어 처리 강화
# - 문자열 -> 토큰 ID 변환, 예시: "AI는 의료 분야에서 활용된다." -> [101, 1234, 5678, ...]
# - 모델 입력: input_ids 토큰 ID 배열, attention_mask 패딩 여부 표시, 모델 출력을 텍스트로 복원 [101, 1234, 5678, ...] -> "AI는 의료 분야에서 활용된다."

# 6. 요약 처리
# - 요약 토크나이저 + 요약 모델 (KoBART 기반)
# - 반복 억제, 길이 조절, 다양성 확보 등 파라미터 튜닝
# - 후처리(clean_summary)로 중복 제거
# - from transformers import BartTokenizer # 영어 전용이라 맞지 않음

# 7. 최종 응답: 외부 서비스 연계 검토
# - Local LLM은 GPU 장비 한계로 로컬 실행은 패스 -> 대신 Copilot에 문의하여 자연스러운 답변 재구성
# - 검토: 현재 로컬 GPU 장비로는 생성형 LLM 모델을 도메인에 맞추어 파인튜닝은 불가능, 현재 추론 모델도 좋은 성능의 모델로 교체 불가능

# 8. 서비스: 구글, 네이버 RAG 시스템 구성
# - /llm_app/transformer_rag3_24_app.py
# - FastAPI 구동 정보: 터미널에서 구동, uvicorn transformer_rag3_24_app:app --reload, 경로 포함 uvicorn LLM.llm_app.transformer_rag3_24_app:app --reload
# - FastAPI 서비스: /search, 입력: 질의문, 출력: QA + 요약 결과 + 출처 정보
# - 1)  [사용자 질의]
# - 2)  [FastAPI 엔드포인트: /search]
# - 3)  [SentenceTransformer: 임베딩]
# - 4)  [Qdrant 의미 기반 검색]
# - 5)  [검색 결과 문서]
# - 6)  [KoELECTRA QA 모델 + MeCab 후처리(형태소 분석: 한국어 처리 강황)]
# - 7)  [KoBART Summarization 모델 + clean_summary]
# - 8)  [응답 + 출처 표시]
# - 9)  [최종 응답 LLM 모델은  외부 서비스 연계 검토: 자연스러운 문장]
# - 10) [최종 사용자 응답]

In [7]:
# 데이터 수집 모듈 구현
import feedparser               # RSS 피드를 파싱해서 뉴스 항목을 가져옴
from bs4 import BeautifulSoup   # HTML 태그 제거 및 텍스트 정제
from datetime import datetime
import psycopg2                 # PostgreSQL DB 연결 및 SQL 실행
import logging                  # 실행 과정 기록 (파일 + 콘솔 출력)

# RSS 가져오기 함수
def fetch_rss(rss_url: str):
    # feedparser 파싱 -> 뉴스 항목(entries) 리스트 반환
    feed = feedparser.parse(rss_url)
    return feed.entries

# 뉴스 파싱 함수
def parse_entry(entry):
    title = entry.title # 뉴스 제목, RSS 항목의 <title> 태그에 해당
    content_raw = entry.description if "description" in entry else ""  # 뉴스 본문, RSS 항목에 description 속성, 없으면 빈 문자열 반환
    content = BeautifulSoup(content_raw, "html.parser").get_text() # 텍스트만 추출, HTML 제거
    url = entry.link # 뉴스 원문 링크(URL), RSS 항목의 <link> 태그
    published_at = datetime(*entry.published_parsed[:6]) if hasattr(entry, "published_parsed") else datetime.now() # entry.published_parsed가 있으면 datetime 객체로 변환(연, 월, 일, 시, 분, 초)
    source_name = entry.source.title if hasattr(entry, "source") else "Google News" # entry.source가 있으면 그 안의 title을 사용하고, 없으면 기본값 "Google News"로 설정
    source_url = entry.source.href if hasattr(entry, "source") else "" # entry.source가 있으면 href 속성을 사용하고, 없으면 빈 문자열로 처리

    return title, content, url, published_at, source_name, source_url

# DB Insert 함수
def insert_article(cur, conn, title, content, url, published_at, source_name, source_url):
    try: # SQL 실행
        cur.execute("""
            INSERT INTO news_articles (title, content, url, published_at, source_name, source_url)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (url) DO NOTHING;
        """, (title, content, url, published_at, source_name, source_url))
        if cur.rowcount > 0: # cur.rowcount → SQL 실행 후 영향을 받은 행 수
            logging.info("데이터 Insert 성공: %s", title) # 1 이상이면 새로운 데이터가 추가된 것 → “Insert 성공” 로그 기록
        else:
            logging.info("중복으로 건너뜀: %s", title) # 0이면 중복으로 인해 추가되지 않은 것 → “중복으로 건너뜀” 로그 기록
        conn.commit() # 실행 결과를 실제 DB 반영
    except Exception as e:
        logging.error("Insert 실패: %s, 에러: %s", title, e)

# 로깅 설정: 파일 + 콘솔 출력
logging.basicConfig(
    level=logging.INFO, # INFO 이상 레벨 기록
    format="%(asctime)s [%(levelname)s] %(message)s", # 출력 형식
    handlers=[
        logging.FileHandler("24.transformer_rag3_app.log", encoding="utf-8"), # 파일 저장
        logging.StreamHandler() # 콘솔 출력
    ]
)
    
# RSS 함수 호출
rss_url = "https://news.google.com/rss?hl=ko&gl=KR&ceid=KR:ko"
entries = fetch_rss(rss_url)

# DB 연결
conn = psycopg2.connect(dbname="newsdb", user="newsuser", password="1234", host="localhost", port="5432")
cur = conn.cursor()

# 뉴스 파싱 함수 -> DB Insert 함수
for entry in entries:
    title, content, url, published_at, source_name, source_url = parse_entry(entry)
    insert_article(cur, conn, title, content, url, published_at, source_name, source_url)

cur.close()
conn.close()
logging.info("DB 연결 종료 완료")

2026-05-14 10:31:06,000 [INFO] 데이터 Insert 성공: 광주 고교생 살해 피의자 신상공개…23살 장윤기 - 한겨레
2026-05-14 10:31:06,006 [INFO] 중복으로 건너뜀: "트럼프 대통령 9년 만에 베이징 방문…'무역·반도체·이란 전쟁' 등 현안 산적" - VOA 한국어 홈페이지
2026-05-14 10:31:06,009 [INFO] 중복으로 건너뜀: 청와대, 김용범이 꺼낸 반도체 초과세수 활용방안 본격 검토 착수 - 한겨레
2026-05-14 10:31:06,018 [INFO] 데이터 Insert 성공: 윤석열 ‘내란 우두머리’ 2심 첫 공판···법관기피 신청 ‘변수’ - 경향신문
2026-05-14 10:31:06,024 [INFO] 중복으로 건너뜀: [단독]“돈 없을 것 같아서”···‘네팔 치매 노모’ 비자 변경 거절한 법무부 - 경향신문
2026-05-14 10:31:06,057 [INFO] 중복으로 건너뜀: 갑자기 여름 시작?…오늘 서울 낮 기온 31도 - 한겨레
2026-05-14 10:31:06,065 [INFO] 데이터 Insert 성공: 삼성, 유니폼 착용하고 떠난 초등생 비보에 "슬프고 안타깝다" - 한국경제
2026-05-14 10:31:06,072 [INFO] 데이터 Insert 성공: 민주 6선 조정식, 후반기 국회의장 후보 선출…이 대통령 정무특보 출신 - 한겨레
2026-05-14 10:31:06,080 [INFO] 데이터 Insert 성공: 野 "여종업원 외박 강요하다 주폭"… 정원오 측 "사실 아닌 일방적 주장" - 조선일보
2026-05-14 10:31:06,085 [INFO] 데이터 Insert 성공: 이 대통령, 베선트에 ‘통화 스와프’ 제안…“핵심 광물, 외환시장 협력 필요” - 한겨레
2026-05-14 10:31:06,091 [INFO] 데이터 Insert 성공: 아내에 뺨 맞았던 마크롱…"이란 여배우 때문이었다" - 조선일보
2026-05-14 10:31:06,100 [INF

In [8]:
# 뉴스 조회
import psycopg2
import logging

# DB 연결
conn = psycopg2.connect(dbname="newsdb", user="newsuser", password="1234", host="localhost", port="5432")
cur = conn.cursor()

# 1. 전체 뉴스 개수 조회
try:
    cur.execute("""
            SELECT COUNT(*) 
            FROM news_articles;
    """)
    count = cur.fetchone()[0]
    logging.info("총 뉴스 개수: %s", count)
except Exception as e:
    logging.error("총 뉴스 개수 중 에러 발생:", e)

# 2. 최신 뉴스 5개 조회
try:
    cur.execute("""
            SELECT title, published_at
            FROM news_articles
            ORDER BY published_at DESC
            LIMIT 5;
    """)
    lastest_news = cur.fetchall()
    logging.info("최신 뉴스 5개:")
    for row in lastest_news:
        logging.info("제목: %s | 발행일: %s", row[0], row[1])
except Exception as e:
    logging.error("최신 뉴스 검색 중 에러 발생:", e)

# 3. 특정 키워드 검색(예시: AI)
try:
    keyword = 'AI'
    cur.execute("""
                SELECT title, url
                FROM news_articles
                WHERE content LIKE %s
                ORDER BY published_at DESC
                LIMIT 10;
    """, (f"%{keyword}%",)) # 튜플 형식 지켜야 에러가 안남
    keyword_news = cur.fetchall()
    logging.info("키워드 '%s' 관련 뉴스:", keyword)
    for row in keyword_news:
        logging.info("제목: %s | URL: %s", row[0], row[1])
except Exception as e:
    logging.error("키워드 검새 중 에러 발생:", e)

# 4. 출처별 뉴스 개수
try:
    cur.execute(""" 
            SELECT source_name, COUNT(*)
            FROM news_articles
            GROUP BY source_name
            ORDER BY COUNT(*) DESC;
    """)
    source_stats = cur.fetchall()
    logging.info("출처별 뉴스 개수:")
    for row in source_stats:
        logging.info("출처: %s | 개수: %s", row[0], row[1])
except Exception as e:
    logging.error("출처별 뉴스 개수 검색 중 에러 발생:", e)

cur.close()
conn.close()
logging.info("조회 완료 및 DB 연결 종료")

2026-05-14 10:31:16,245 [INFO] 총 뉴스 개수: 365
2026-05-14 10:31:16,250 [INFO] 최신 뉴스 5개:
2026-05-14 10:31:16,251 [INFO] 제목: 삼성스토어 홍대에 브롤스타즈 뜬다 … ‘스타 노바’ 팝업스토어 오픈 - 경향게임스 | 발행일: 2026-05-14 01:13:21
2026-05-14 10:31:16,252 [INFO] 제목: '워해머 스컬즈' 10주년, 역대급 대작 라인업 공개 예고 … '스페이스 마린2', '다크타이드' 등 신규 정보 공개 - 경향게임스 | 발행일: 2026-05-14 01:11:27
2026-05-14 10:31:16,254 [INFO] 제목: BTS에 중남미 열광...새 음반 스트리밍 수치 한국 넘어서 - YTN | 발행일: 2026-05-14 00:54:00
2026-05-14 10:31:16,256 [INFO] 제목: 코스피, 장 초반 7900선 돌파 ‘8000 시도’…삼성전자는 30만원 눈앞 - 조선일보 | 발행일: 2026-05-14 00:36:00
2026-05-14 10:31:16,257 [INFO] 제목: 청와대, 김용범이 꺼낸 반도체 초과세수 활용방안 본격 검토 착수 - 한겨레 | 발행일: 2026-05-14 00:24:00
2026-05-14 10:31:16,263 [INFO] 키워드 'AI' 관련 뉴스:
2026-05-14 10:31:16,267 [INFO] 제목: 청와대, 김용범이 꺼낸 반도체 초과세수 활용방안 본격 검토 착수 - 한겨레 | URL: https://news.google.com/rss/articles/CBMia0FVX3lxTE1qU043VVdnRnotWDJLT2xQOGxLMjlWcXVmVE4tTFBaaXU1cUhkQWZHcXE0Q3UtVzc2LVd6NW1hZTQ1NFlmUTBCc0lVc210eVJ4ZVJTeEkxd2VNQmdSMFJqeTAwNnJvSVdfRmsw?oc=5
2026-05-14 10:31:16,273 [INFO] 

In [9]:
#  색인용 데이터 추출
import psycopg2
import logging

# DB 연결
conn = psycopg2.connect(
    dbname="newsdb", 
    user="newsuser", 
    password="1234", 
    host="localhost", 
    port="5432"
)

# cursor 객체 생성
cur = conn.cursor()

try:
    # 색인용 데이터 조회, 최근 데이터 100개 기준
    cur.execute(""" 
                SELECT id, title, content, url, published_at, source_name
                FROM news_articles
                ORDER BY published_at DESC
                LIMIT 100;
    """)
    articles = cur.fetchall()
    logging.info("색인용 데이터 %s개 가져옴:", len(articles))

    # Qdrant에 넣을 준비: 리스트 형태로 변환
    index_data = []
    for row in articles:
        record = {
            "id": row[0],
            "title": row[1],
            "content": row[2],
            "url": row[3],
            "published_at": row[4],
            "source_name": row[5]
        }
        index_data.append(record)

except Exception as e:
    logging.error("색인용 조회 에러 발생: %s", e)

# DB connect 종료
cur.close()
conn.close()
logging.info("색인 데이터 조회 완료 및 DB 연결 종료")

2026-05-14 10:31:26,763 [INFO] 색인용 데이터 100개 가져옴:
2026-05-14 10:31:26,766 [INFO] 색인 데이터 조회 완료 및 DB 연결 종료


In [10]:
# Qdrant Collection 생성
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import logging

# Qdrant 연결
qdrant = QdrantClient(host="localhost", port=6333)

# Qdrant Collection 초기화 + 생성
# qdrant.recreate_collection( # 지정한 이름의 컬렉션을 새로 생성, 이미 같은 이름의 컬렉션이 있으면 삭제 후 다시 생성(초기화 + 생성 기능)
#     collection_name="news_articles", # 컬렉션 명
#     vectors_config=VectorParams( # 벡터 설정 정의
#         size=384, # 모델 MiniLM-L12-v2 임베딩 차원 수(384차원)
#         distance=Distance.COSINE # 벡터 유사도 계산 방식(COSINE 추천, 의미 유사도 검색에 가장 많이 사용), DOT(내적 기반), EUCLID(거기 기반)
#     )
# )

# Qdrant Collecton 생성
if not qdrant.collection_exists("news_articles"):
    qdrant.create_collection(
        collection_name="news_articles", # 컬렉션 명
        vectors_config=VectorParams( # 벡터 설정 정의
            size=384, # 모델 MiniLM-L12-v2 임베딩 차원 수(384차원)
            distance=Distance.COSINE # 벡터 유사도 계산 방식(COSINE 추천, 의미 유사도 검색에 가장 많이 사용), DOT(내적 기반), EUCLID(거기 기반)
        )
    )
    logging.info(f"Qdrant Collection 'news_articles' 생성 완료")
else:
    logging.info(f"Qdrant Collection 'news_articles' 존재 합니다.")

2026-05-14 10:31:33,153 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-05-14 10:31:33,196 [INFO] HTTP Request: GET http://localhost:6333/collections/news_articles/exists "HTTP/1.1 200 OK"
2026-05-14 10:31:33,200 [INFO] Qdrant Collection 'news_articles' 존재 합니다.


In [11]:
# Qdrant news_articles 컬렉션 update/insert
import torch
from sentence_transformers import SentenceTransformer # Hugging Face sentence-transformer 라이브러리 사용
from qdrant_client import QdrantClient
from qdrant_client.http import models # qdrant 컬렉션 및 검색 관련 설정 정의하는 데이터 모델(구조체) 로드
import logging

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
logging.info(f"Pytorch Version : {torch.__version__}, Device : {device}")

# 다국어 임베딩 모델 로드
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Qdrant 서버 연결
qdrant = QdrantClient(host="localhost", port=6333)

# 임베딩 함수 생성
def generate_embedding(text: str):
    # 본문(content)을 벡터로 변환
    embedding = model.encode(text)
    return embedding.tolist() # Qdrant에 넣기 위해 list 변환

# 데이터 저장 형식 설명
# qdrant.upsert()는 삽입(insert)+업데이트(update) 기능을 동시에 수행, 같은 ID가 있으면 덮어쓰고, 없으면 새로 추가한다
    # { 
    #     "id": 1,
    #     "vector": [0.123, -0.456, 0.789, ...],
    #     "payload": {
    #         "title": "AI 의료 활용",
    #         "content": "AI가 의료 분야에서 활용되는 사례...",
    #         "date": "2026-03-11",
    #         "author": "홍길동"
    #     }
    # }
# Qdrant update/insert 함수
def insert_to_qdrant(data):
    ids = []
    vectors = []
    payloads = []
    
    for item in data:
        # 제목 + 내용 임베딩(벡터 변환) 대상 데이터
        text = item["title"] + " " + item["content"]
        # 임베딩 함수 호출
        embedding = generate_embedding(text=text)

        ids.append(item["id"]) # ID 값
        vectors.append(embedding) # 임베딩 처리 후 벡터 값
        payloads.append(item) # 원본 데이터
    
    qdrant.upsert(
        collection_name="news_articles",
        points=models.Batch(
            ids=ids,
            vectors=vectors,
            payloads=payloads # paylods(JSON) 형태로 저장
        )
    )
    logging.info("데이터 임베딩 및 Qdrant 저장 완료")
    

# Batch 단위로 Qdrant update/insert
batch_size = 10
total = len(index_data)

for i in range(0, len(index_data), batch_size):
    chunk = index_data[i : i + batch_size] # 2개씩 잘라서 가져온다
    insert_to_qdrant(chunk) # 잘라낸 청크를 Qdrant에 전달

    # 진행 상황 계산
    start_id = i
    end_id = i + len(chunk) - 1
    processed = end_id + 1
    progress = (processed / total) * 100
    remaining = total - processed

    logging.info(
        f"Batch {start_id} ~ {end_id} 인덱싱 완료" 
        f"(총 {len(chunk)}건), 진행률 : {progress:.2f}%, 남은 데이터 : {remaining}건"
    )

# 최종 데이터 개수 확인
count_result = qdrant.count(collection_name="news_articles")
logging.info(f"Qdrant news_articles 컬렉션 최종 저장 건수 : {count_result}건")


2026-05-14 10:32:27,038 [INFO] Pytorch Version : 2.2.2, Device : cpu
2026-05-14 10:32:27,163 [INFO] Use pytorch device_name: cpu
2026-05-14 10:32:27,165 [INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
2026-05-14 10:32:32,853 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-14 10:32:53,224 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_articles/points?wait=true "HTTP/1.1 200 OK"
2026-05-14 10:32:53,229 [INFO] 데이터 임베딩 및 Qdrant 저장 완료
2026-05-14 10:32:53,231 [INFO] Batch 90 ~ 99 인덱싱 완료(총 10건), 진행률 : 100.00%, 남은 데이터 : 0건
2026-05-14 10:32:53,257 [INFO] HTTP Request: POST http://localhost:6333/collections/news_articles/points/count "HTTP/1.1 200 OK"
2026-05-14 10:32:53,267 [INFO] Qdrant news_articles 컬렉션 최종 저장 건수 : count=184건


In [12]:
# Qdrant 검색(Query)

# Qdrant 검색 함수
def search_qdrant(query: str, top_k: int = 5):
    # 질의 임베딩 생성
    query_vector = generate_embedding(query)

    # qdrant에서 검색
    result = qdrant.query_points(
        collection_name="news_articles",
        query=query_vector,
        limit=top_k
    )
    # result → QueryResponse 객체, 검색 결과를 감싸는 객체
    # result.points → ScoredPoint 리스트 : 실제 결과는 result.points에 들어 있는 ScoredPoint 리스트, score, .payload 등을 꺼내 쓰면 된다
    return result.points # ScoredPoint 리스트 반환

# qdrant 검색 함수 호출
search_result = search_qdrant("AI 의료 활용 사례")

# 결과 출력
for r in search_result:
    logging.info(
        f"Score : {r.score:.4f}, Title : {r.payload.get('title')}, "
        f"Source : {r.payload.get('source_name')} , URL : {r.payload.get('url')} "
    )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-14 10:33:18,231 [INFO] HTTP Request: POST http://localhost:6333/collections/news_articles/points/query "HTTP/1.1 200 OK"
2026-05-14 10:33:18,236 [INFO] Score : 0.3509, Title : ‘절단’ 대신 ‘재건’… 명지병원, 당뇨발 ‘비절단 미세재건’ 치료 강화 - 후생신보, Source : 후생신보 , URL : https://news.google.com/rss/articles/CBMiREFVX3lxTFBsWHhNQ0t5RVRtRk84eGgyczdnNXpFdU9DcF9KMFFtaldNZmVKZXhBcmxURllzNXlZX3hhTnRUcGtWVnha?oc=5 
2026-05-14 10:33:18,237 [INFO] Score : 0.3211, Title : "게이밍 넘어 전문가로" AMD, 3D V-캐시 탑재 라이젠 프로 시리즈 발표 - 디지털투데이, Source : 디지털투데이 , URL : https://news.google.com/rss/articles/CBMic0FVX3lxTE9uRGRJOFlWbW1Ib0xpVGo5THBoYkpyTW9Zd3NHTjJmNHVYTXRjV3VzeHIyRk04dTMyc2NuZFo0WXBDOXVuZVBKWXRadVdwcnRnTmw3X0t3cVZzVUtrbFBib2FnTURPRmdvaFU5ZnpLRDA4Zmc?oc=5 
2026-05-14 10:33:18,239 [INFO] Score : 0.3185, Title : 안철수 “기업 수익 국가가 나누는 건 공산주의” - 경향신문, Source : 경향신문 , URL : https://news.google.com/rss/articles/CBMiWkFVX3lxTE1SMkZCVlQwaWRlR1B0OHgxUUZXRnpFUDVRT2g4VmYyeFNCYTFIOXdhSVRkclVIZ3dGQy04OVBhakI1Q3lxOWtqSlFJVlo3N1BkNG1lVmZx

In [13]:
# 형태소 분석 테스트

# MeCab 설치 (macOS 기준) : brew install mecab mecab-ipadic
# 파이썬 바인딩 설치 : pip install mecab-python3

# 형태소 분석 테스트
from mecab import MeCab

mecab = MeCab()
print(mecab.morphs("인공지능은 의료 분야에서 활용된다."))
print(mecab.nouns("인공지능은 의료 분야에서 활용된다."))
print(mecab.pos("인공지능은 의료 분야에서 활용된다."))

['인공지능', '은', '의료', '분야', '에서', '활용', '된다', '.']
['인공지능', '의료', '분야', '활용']
[('인공지능', 'NNP'), ('은', 'JX'), ('의료', 'NNG'), ('분야', 'NNG'), ('에서', 'JKB'), ('활용', 'NNG'), ('된다', 'XSV+EF'), ('.', 'SF')]


In [ ]:
# QA 모델 적용 : embedding + qdrant + QA Model
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from mecab import MeCab
import gc

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
logging.info(f"PyTorch Version : {torch.__version__}, Device : {device}")

# embedding
embedder_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder_model = SentenceTransformer(embedder_name)

# KorQuAD (Korean Question Answering Dataset) : 한국어 위키 문서를 기반으로 만든 QA 데이터셋으로 질문과 답변이 문맥 내에서 매핑되어 있다
qa_model_name = "monologg/koelectra-base-v3-finetuned-korquad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name) # QA Tokenizer
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name) # QA Model

# embedding 함수
def generate_embedding(text: str):
    embedding = embedder_model.encode(text) # 임베딩 벡터 변환
    return embedding.tolist()

# qdrant 검색 함수
def search_qdrant(query: str, top_k: int = 5):
    # 질의 임베딩 생성
    query_vector = generate_embedding(query)

    # qdrant 검색
    query_result = qdrant.query_points(
        collection_name="news_articles",
        query=query_vector,
        limit=top_k
    )
    # query_result : QueryResponse 객체, 검색 결과를 감싸는 객체
    # query_result.points : ScoredPoint 리스트 실제 결과는 query_result.points에 들어 있는 ScoredPoint 리스트, score, .payload 등을 꺼내 쓰면 된다
    return query_result.points

# QA 모델 파이프라인: 
# QA 토크나이저 > QA 모델 > 답변 시작/끝 위치 > 토큰 ID를 토큰 문자열 변환 > 특수 토큰 제거 > 토큰을 문자열로 변환
def qa_pipeline(question, context):
    # 토크나이징: 질문과 문서를 토큰 ID로 변환
    inputs = qa_tokenizer(
        question, # 질문
        context, # 문서
        return_tensors="pt" # 파이토치 형태로 리턴
    ).to(device)

    # QA 모델 추론
    with torch.no_grad():
        # QA 모델 추론
        outputs = qa_model(**inputs) # **inputs 파라미터 형태로 입력
    
    # 후처리 1. 답변 시작/끝 위치 예측하는 확률 분포
    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1
    logging.info(f"answer_start : {answer_start}")
    logging.info(f"answer_end : {answer_end}")

    # 후처리 2. 시작/끝 위치를 잡을때, 최소/최대 길이를 강제로 설정(최소 5토큰 이상, 최대 50토큰 이하)
    if answer_end - answer_start < 5:
        answer_end = answer_start + 5
    elif answer_end - answer_start > 50:
        answer_end = answer_start + 50
    logging.info(f"answer_start 조건 추가 : {answer_start}")
    logging.info(f"answer_end 조건 추가 : {answer_end}")

    # 후처리 3. 토큰ID > 토큰 문자열 변환
    tokens = qa_tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0][answer_start:answer_end]
    )
    logging.info(f"tokens 토큰ID > 토큰 문자열 변환 : {tokens}")

    # 후처리 4. [CLS],[SEP],[PAD] 특수 토큰 제거
    tokens = [t for t in tokens if t not in qa_tokenizer.all_special_tokens]
    logging.info(f"tokens [CLS],[SEP],[PAD] 특수 토큰 제거 : {tokens}")

    # 후처리 5. 토큰 > 최종적으로 사람이 읽을 수 있는 문자열로 복원
    answer = qa_tokenizer.convert_tokens_to_string(tokens)
    logging.info(f"토큰 > 최종적으로 사람이 읽을 수 있는 문자열로 복원 : {answer}")

    return answer

# 형태소 분석
mecab = MeCab()
# 후처리 함수
def clean_result(text: str):
    text = text.strip() # 공백 제거
    text = text.replace("[CLS]", "").replace("[SEP]", "").replace("[PAD]", "") # 특수문자 토큰 제거

    morphs = mecab.pos(text)
    # 변형할 품사가 없으면 원본 반환
    if not morphs:
        return text
    
    # 마지막 단어/품사
    last_word, last_pos = morphs[-1]

    # 동사/형용사 처리
    if last_pos.startswith("VV") or last_pos.startswith("VA") or "XSV" in last_pos:
        if not text.endswith("다."):
            if text.endswith("다"):
                text += "."
            else:
                text += "다."
    
    # 명사 처리
    elif last_pos.startswith("NN"):
        if not text.endswith("이다."):
            text += "이다."
    
    # 조사로 끝나는 경우 -> 직전 형태소 확인
    elif last_pos.startswith("J"):
        prev_word, prev_pos = morphs[-2]
        if prev_pos.startswith("NN"): # 명사
            text += "이다."
        elif prev_pos.startswith("VV") or prev_pos.startswith("VA"): # 동사/형용사
            text += "다."
    
    # 기본 처리
    else:
        if not text.endswith("다."):
            if text.endswith("다"):
                text += "."
            else:
                text += "다."
    
    return text

# qdrant 검색 결과
query = "AI 의료 활용 사례"
search_result = search_qdrant(query)
for r in search_result:
    logging.info(
        f"Score: {r.score:.4f}, "
        f"Title: {r.payload.get('title')}, "
        f"Source: {r.payload.get('source_name')}, "
        f"URL: {r.payload.get('url')}, "
        f"Content: {r.payload.get('content')}"
    )

# 검색된 문서들을 context로 합치기
contexts = [r.payload.get('content') for r in search_result]
# 여러 문서들을 하나의 문자열로 합친다
context = " ".join(contexts)
logging.info(f"context : {context}")

# QA 모델: 질문(question)/문서(context)를 입력으로 받아서 답변을 복원 또는 생성
qa_answers = []
for i, doc in enumerate(contexts):
    # qa 모델 파이프라인(질문, 문서)
    ans = qa_pipeline(question=query, context=doc)

    if ans and ans not in ["[CLS]", ""]:
        qa_answers.append(f"{i+1} {clean_result(ans)}")
logging.info(f"qa_answers 형태소 분석 포함 후처리 : {qa_answers}")

# 최종 답변 선택
if qa_answers:
    final_qa_answers = qa_answers
else:
    # QA 모델 답변 실패시 검색 문서 전체를 fallback으로 사용
    final_qa_answers = context
logging.info(f"질문 : {query}")
logging.info(f"답변 : {final_qa_answers}")

# 메모리 정리: 객체 삭제, 메모리 반환
del qa_tokenizer
del qa_model
del embedder_model

gc.collect()
# GPU cuda 사용시 메모리 반환
if device=="cuda":
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

2026-05-14 14:30:23,351 [INFO] PyTorch Version : 2.2.2, Device : cpu
2026-05-14 14:30:23,357 [INFO] Use pytorch device_name: cpu
2026-05-14 14:30:23,363 [INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-14 14:30:30,250 [INFO] HTTP Request: POST http://localhost:6333/collections/news_articles/points/query "HTTP/1.1 200 OK"
2026-05-14 14:30:30,254 [INFO] Score: 0.3509, Title: ‘절단’ 대신 ‘재건’… 명지병원, 당뇨발 ‘비절단 미세재건’ 치료 강화 - 후생신보, Source: 후생신보, URL: https://news.google.com/rss/articles/CBMiREFVX3lxTFBsWHhNQ0t5RVRtRk84eGgyczdnNXpFdU9DcF9KMFFtaldNZmVKZXhBcmxURllzNXlZX3hhTnRUcGtWVnha?oc=5, Content: ‘절단’ 대신 ‘재건’… 명지병원, 당뇨발 ‘비절단 미세재건’ 치료 강화  후생신보명지병원, 당뇨발 '비절단 미세재건' 본격화  헬스미디어뉴스당뇨발 전세계 평생유병률 6.3%…전기자극치료로 상처치유 위한 미세환경 회복  네이트명지병원 당뇨발 클리닉, ‘비절단 미세재건’ 본격화...통합진료체계 강화  라포르시안
2026-05-14 14:30:30,257 [INFO] Score: 0.3211, Title: "게이밍 넘어 전문가로" AMD, 3D V-캐시 탑재 라이젠 프로 시리즈 발표 - 디지털투데이, Source: 디지털투데이, URL: https://news.google.com/rss/articles/CBMic0FVX3lxTE9uRGRJOFlWbW1Ib0xpVGo5THBoYkpyTW9Zd3NHTjJmNHVYTXRjV3VzeHIyRk04dTMyc2NuZFo0WXBDOXVuZVBKWXRadVdwcnRnTmw3X0t3cVZzVUtrbFBib2FnTURPRmdvaFU5ZnpLRDA4Zmc?oc=5, Content: "게이밍 넘어 전문가로" AMD, 3D V-캐시 탑재 라이젠 프로 시리즈 발표  디지털투데이Google 뉴스에서 헤드라인 및 의견 더보기
2026-05-14